
# Paper 2 — Goal 3
## Notebook 34: Goal 2 → Goal 3 frozen model-bundle bridge

**Purpose.** This notebook is the mandatory bridge between the sealed Goal 2 analysis and the controlled Goal 3 perturbation experiment.

It does **not** apply any perturbation and does **not** inspect controlled Goal 3 clinical responses.

It performs exactly five jobs:

1. re-assert the frozen Goal 2 / Goal 3 prerequisite seals;
2. reconstruct the **first outer repeat only** (`repeat = 1`) using the frozen five-fold participant split;
3. rebuild the authoritative fold-specific `M_A`, `M_A+Q`, and protocol-concordant `M_A-resQ` ridge models from **unmodified outer-training observations only**;
4. serialize the complete inference state needed by the later perturbation notebook, including training-only A/Q preprocessing, Q→A residualization, final scaling, outcome models, and task/fold-specific fixed QCHAN references;
5. reproduce the authoritative Goal 2 repeat-1 held-out predictions to a strict numerical tolerance before writing the bundle seal.

### Frozen model ladder

\[
M_A = \mathrm{Age}+A
\]

\[
M_{A+Q} = \mathrm{Age}+A+Q
\]

\[
M_{A-resQ} = \mathrm{Age}+A_{resQ},
\qquad
A_{resQ}=A-\hat g_{\mathrm{train}}(Q)
\]

The Q→A map for `M_A-resQ` is the corrected, protocol-concordant map selected inside outer-training data in Goal 2. It is never refit on held-out data.

### Critical rule for Goal 3

The later perturbation notebook must feed remeasured Q/A from a held-out perturbed waveform through the **already frozen bundle for that held-out participant's fold**. Perturbed audio must never be used for fitting, tuning, scaling, QCHAN-reference construction, or Q→A residualizer fitting.

### Run instruction

Use **Kernel → Restart Kernel and Run All Cells**.

If any reproduction gate fails, **stop**. Do not loosen tolerances and do not proceed to the controlled perturbation notebook until the discrepancy is resolved.


**v1.0.1 patch:** accepts the governed Stage-B seal status `GOAL3_STAGE_B_SIGNAL_ONLY_SEALED`. No scientific/modeling logic changed.

In [1]:

from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import sys
import warnings
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# Frozen bridge configuration
# ---------------------------------------------------------------------

ENGINE_VERSION = "goal3-goal2-model-bundle-bridge-v1.0.0"
PAPER1_COMMIT = "cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8"

OUTER_REPEAT = 1
OUTER_FOLDS = 5
MODELS = ["M_A", "M_A+Q", "M_A-resQ"]

# Strict but realistic same-environment numerical reproduction tolerance.
PRED_ATOL = 1e-10
PRED_RTOL = 1e-10

RESET_OUTPUTS = False

EXPECTED = {
    "recordings": 519,
    "participants": 224,
    "als_participants": 158,
    "control_participants": 66,
    "als_recordings": 418,
    "control_recordings": 101,
    "diagnosis_age_complete_participants": 199,
    "diagnosis_age_complete_als": 158,
    "diagnosis_age_complete_controls": 41,
    "severity_participants_60d": 145,
    "severity_pairs_60d": 398,
    "primary_A": 6,
}

def find_project_root() -> Path:
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if (root / "data" / "processed").exists():
            return root
        raise FileNotFoundError(f"Invalid PAPER2_ROOT: {root}")

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data" / "processed").exists()
            and (candidate / "data" / "manifests").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the Paper 2 repository root. "
        "Run this notebook from inside the Paper_2_Leakage/Code repository "
        "or set PAPER2_ROOT."
    )

ROOT = find_project_root()
PROCESSED = ROOT / "data" / "processed"
MANIFESTS = ROOT / "data" / "manifests"
EXTERNAL = ROOT / "external" / "quality_framework_features"

OUT = (
    ROOT / "outputs" / "goal3"
    / "stageD_goal2_model_bundle_bridge_v1_0" / "final"
)
TABLES = OUT / "tables"
AUDIT = OUT / "audit"
BUNDLES = OUT / "bundles"
QCHAN_REFS = OUT / "qchan_references"

for directory in [OUT, TABLES, AUDIT, BUNDLES, QCHAN_REFS]:
    directory.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

def stable_hash(payload) -> str:
    text = json.dumps(
        payload, sort_keys=True, default=str, separators=(",", ":")
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def atomic_json(payload: dict, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def atomic_csv(frame: pd.DataFrame, path: Path, *, allow_empty=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{path.name}: expected DataFrame.")
    if frame.empty and not allow_empty:
        raise ValueError(f"{path.name}: refusing to write empty table.")
    tmp = path.with_name(f".{path.name}.tmp")
    frame.to_csv(tmp, index=False)
    if not tmp.exists() or tmp.stat().st_size == 0:
        raise IOError(f"{path.name}: temporary CSV is empty.")
    os.replace(tmp, path)

def atomic_joblib(payload, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp")
    joblib.dump(payload, tmp, compress=3)
    if not tmp.exists() or tmp.stat().st_size == 0:
        raise IOError(f"{path.name}: temporary joblib is empty.")
    os.replace(tmp, path)

def safe_read_csv(path: Path, required=None) -> pd.DataFrame:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing/empty CSV: {path}")
    frame = pd.read_csv(path, low_memory=False)
    if required:
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    return frame

print("Paper 2 root:", ROOT)
print("Engine:", ENGINE_VERSION)
print("Outer repeat frozen for controlled Goal 3:", OUTER_REPEAT)
print("Prediction reproduction tolerance:", PRED_ATOL, PRED_RTOL)


Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Engine: goal3-goal2-model-bundle-bridge-v1.0.0
Outer repeat frozen for controlled Goal 3: 1
Prediction reproduction tolerance: 1e-10 1e-10



## 1. Hard prerequisite seals

This gate deliberately rechecks the state you already audited manually. It also freezes hashes of the exact inputs that this bridge consumes.

The bridge is blocked unless:

- Goal 2 final completion is sealed;
- Goal 3 Stage A exists;
- Goal 3 Stage B signal-only calibration is sealed;
- the final perturbation manifest contains exactly 18 nonzero dose rows = 6 retained transformations × 3 doses;
- the AM-QADD disposition and QCHAN amendment are present;
- Goal 3 controlled Q/A measurement preflight is sealed.


In [2]:

PATHS = {
    # Frozen cohort / feature inputs
    "recording_table": PROCESSED / "recording_table_phase0.csv",
    "participant_table": PROCESSED / "participant_table.csv",
    "severity_pairs": PROCESSED / "severity_pairs.csv",
    "a_values": PROCESSED / "acoustic_features_frozen.csv",
    "a_registry": MANIFESTS / "a_registry.csv",
    "a_manifest": MANIFESTS / "a_freeze_manifest.json",
    "q_registry": MANIFESTS / "q_registry.csv",
    "split_manifest": MANIFESTS / "split_manifest.csv",
    "qchan_ready": MANIFESTS / "qchan_cache_ready.json",
    "qchan_cache_index": MANIFESTS / "qchan_spectrum_cache_index.csv",

    # Goal 2 authoritative state
    "goal2_done": (
        ROOT / "outputs" / "goal2"
        / "goal2_completion_v1_0" / "final" / "DONE.json"
    ),
    "goal2_dx_oof": (
        ROOT / "outputs" / "goal2"
        / "goal2_completion_v1_0" / "final" / "oof"
        / "goal2_diagnosis_oof_authoritative.csv"
    ),
    "goal2_sev_oof": (
        ROOT / "outputs" / "goal2"
        / "goal2_completion_v1_0" / "final" / "oof"
        / "goal2_severity_oof_authoritative.csv"
    ),
    "goal2_corrected_manifest": (
        ROOT / "outputs" / "goal2"
        / "goal2_completion_v1_0" / "final" / "tables"
        / "goal2_corrected_residualizer_fold_manifest.csv"
    ),
    "goal2_primary_manifest": (
        ROOT / "outputs" / "goal2"
        / "goal2_primary_v1_1" / "final" / "tables"
        / "goal2_fold_model_manifest.csv"
    ),

    # Goal 3 prerequisite state
    "goal3_stageA": (
        ROOT / "outputs" / "goal3"
        / "stageA_v1_0" / "SUCCESS_STAGE_A.json"
    ),
    "goal3_stageB_seal": (
        ROOT / "outputs" / "goal3"
        / "stageB_signal_only_calibration_v1_3" / "final"
        / "GOAL3_STAGE_B_SIGNAL_ONLY_SEAL.json"
    ),
    "goal3_perturbation_manifest": (
        ROOT / "outputs" / "goal3"
        / "stageB_signal_only_calibration_v1_3" / "final" / "tables"
        / "goal3_perturbation_manifest.csv"
    ),
    "goal3_am_qadd_disposition": (
        ROOT / "outputs" / "goal3"
        / "stageB_signal_only_calibration_v1_3" / "final" / "audit"
        / "AM_QADD_SIGNAL_ONLY_DISPOSITION.json"
    ),
    "goal3_qchan_amendment": (
        ROOT / "outputs" / "goal3"
        / "stageB_signal_only_calibration_v1_3" / "final" / "audit"
        / "QCHAN_SIGNAL_ONLY_DESIGN_AMENDMENT.json"
    ),
    "goal3_measurement_preflight": (
        ROOT / "outputs" / "goal3"
        / "stageC_QA_preflight_v1_0" / "PRE_FLIGHT_SUCCESS.json"
    ),
}

missing = [
    name for name, path in PATHS.items()
    if not path.exists() or (path.is_file() and path.stat().st_size == 0)
]
if missing:
    raise FileNotFoundError(
        "Goal 3 bridge prerequisite(s) missing/empty: " + ", ".join(missing)
    )

goal2_done = json.loads(PATHS["goal2_done"].read_text(encoding="utf-8"))
if str(goal2_done.get("status", "")).upper() != "PASS":
    raise RuntimeError("Goal 2 final DONE.json is not status=PASS.")

stageB_seal = json.loads(
    PATHS["goal3_stageB_seal"].read_text(encoding="utf-8")
)
stageB_status = str(stageB_seal.get("status", "")).strip().upper()

# The governed Stage-B seal uses the explicit status string
# "GOAL3_STAGE_B_SIGNAL_ONLY_SEALED".  Accept that exact frozen status,
# while also remaining compatible with older generic seal labels.
accepted_stageB_statuses = {
    "PASS",
    "SEALED",
    "FROZEN",
    "FINAL",
    "GOAL3_STAGE_B_SIGNAL_ONLY_SEALED",
}

if not stageB_status:
    raise RuntimeError("Stage-B seal exists but contains no status field.")

if stageB_status not in accepted_stageB_statuses:
    raise RuntimeError(
        "Unexpected Stage-B seal status: "
        f"{stageB_status!r}. "
        f"Accepted governed statuses: {sorted(accepted_stageB_statuses)}"
    )

preflight = json.loads(
    PATHS["goal3_measurement_preflight"].read_text(encoding="utf-8")
)
if str(preflight.get("status", "")).upper() != "PASS":
    raise RuntimeError("Goal 3 controlled Q/A measurement preflight is not PASS.")

perturbation_manifest = safe_read_csv(
    PATHS["goal3_perturbation_manifest"],
    required=[
        "family", "transform", "dose_label",
        "candidate_value", "candidate_unit",
    ],
)

expected_transforms = {
    "stationary_colored_broadband",
    "uniform_level_shift",
    "smooth_time_varying_gain",
    "RIR_convolution_RMS_matched",
    "upper_band_restriction",
    "symmetric_hard_clipping",
}

if len(perturbation_manifest) != 18:
    raise RuntimeError(
        f"Expected 18 sealed nonzero dose rows; found {len(perturbation_manifest)}."
    )
if set(perturbation_manifest["transform"].astype(str)) != expected_transforms:
    raise RuntimeError(
        "Sealed perturbation transform set changed:\n"
        + str(sorted(set(perturbation_manifest["transform"].astype(str))))
    )
if set(perturbation_manifest["dose_label"].astype(str)) != {"low", "medium", "high"}:
    raise RuntimeError("Stage-B dose labels are not exactly low/medium/high.")
if perturbation_manifest.groupby("transform")["dose_label"].nunique().ne(3).any():
    raise RuntimeError("At least one retained transform does not have exactly three doses.")
if perturbation_manifest["transform"].astype(str).str.contains(
    "amplitude_modulated", case=False, regex=False
).any():
    raise RuntimeError(
        "AM-QADD unexpectedly appears in the sealed final perturbation manifest."
    )

input_hashes = {
    name: sha256_file(path)
    for name, path in PATHS.items()
    if path.is_file()
}

run_contract = {
    "engine_version": ENGINE_VERSION,
    "outer_repeat": OUTER_REPEAT,
    "outer_folds": OUTER_FOLDS,
    "models": MODELS,
    "prediction_atol": PRED_ATOL,
    "prediction_rtol": PRED_RTOL,
    "paper1_commit": PAPER1_COMMIT,
    "input_hashes": input_hashes,
}
RUN_SIGNATURE = stable_hash(run_contract)

signature_path = OUT / "RUN_SIGNATURE.json"
if signature_path.exists() and not RESET_OUTPUTS:
    existing = json.loads(signature_path.read_text(encoding="utf-8"))
    if existing.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Stage-D output belongs to a different input/run contract. "
            "Do not mix outputs. Set RESET_OUTPUTS=True only after deliberate review."
        )

atomic_json(
    {"run_signature": RUN_SIGNATURE, **run_contract},
    signature_path,
)
atomic_json(input_hashes, AUDIT / "goal3_goal2_bridge_input_hashes.json")

print("GOAL 3 BRIDGE PREREQUISITE GATE: PASS")
print("Run signature:", RUN_SIGNATURE[:16])
print("Stage-B transforms:", sorted(expected_transforms))
display(
    perturbation_manifest[
        ["family", "transform", "dose_label", "candidate_value", "candidate_unit"]
    ].sort_values(["family", "transform", "dose_label"])
)


GOAL 3 BRIDGE PREREQUISITE GATE: PASS
Run signature: 13154dc87de7fe2e
Stage-B transforms: ['RIR_convolution_RMS_matched', 'smooth_time_varying_gain', 'stationary_colored_broadband', 'symmetric_hard_clipping', 'uniform_level_shift', 'upper_band_restriction']


,family,transform,dose_label,candidate_value,candidate_unit
2,QADD,stationary_colored_broadband,high,10.000,dB injected SNR
0,QADD,stationary_colored_broadband,low,35.000,dB injected SNR
1,QADD,stationary_colored_broadband,medium,30.000,dB injected SNR
5,QCHAN,upper_band_restriction,high,2000.000,low-pass cutoff Hz
3,QCHAN,upper_band_restriction,low,4500.000,low-pass cutoff Hz
4,QCHAN,upper_band_restriction,medium,3500.000,low-pass cutoff Hz
8,QDIST,symmetric_hard_clipping,high,0.010,target changed channel-sample fraction
6,QDIST,symmetric_hard_clipping,low,0.001,target changed channel-sample fraction
7,QDIST,symmetric_hard_clipping,medium,0.003,target changed channel-sample fraction
11,QGAIN,smooth_time_varying_gain,high,24.000,dB modulation amplitude



## 2. Load the authoritative Goal 2 inputs and exact frozen feature contracts

This cell reconstructs the same frozen Primary-A and Core-Q contracts used by the authoritative Goal 2 completion.

QCHAN is not read as a globally fixed covariate for prediction. Its four reference-relative features are recomputed fold-safely from the reference-independent spectrum cache.


In [3]:

recording_table = safe_read_csv(
    PATHS["recording_table"],
    required=[
        "participant_id", "logical_recording_id",
        "diagnosis", "age_at_recording_years",
    ],
)
participant_table = safe_read_csv(
    PATHS["participant_table"],
    required=["participant_id", "diagnosis"],
)
severity_pairs_raw = safe_read_csv(
    PATHS["severity_pairs"],
    required=[
        "participant_id", "logical_recording_id",
        "bulbar_score", "abs_delta_days", "within_60_days",
    ],
)
a_values = safe_read_csv(
    PATHS["a_values"],
    required=["participant_id", "logical_recording_id"],
)
a_registry = safe_read_csv(
    PATHS["a_registry"],
    required=[
        "feature", "final_role", "final_transform",
        "support_indicator_column", "a_freeze_version",
    ],
)
q_registry = safe_read_csv(
    PATHS["q_registry"],
    required=["feature", "family", "paper2_role", "transform"],
)
split_manifest = safe_read_csv(
    PATHS["split_manifest"],
    required=["participant_id", "repeat", "outer_fold"],
)
qchan_cache_index = safe_read_csv(
    PATHS["qchan_cache_index"],
    required=[
        "logical_recording_id", "status",
        "spectrum_sha256", "cache_path",
    ],
)
primary_manifest = safe_read_csv(
    PATHS["goal2_primary_manifest"],
    required=[
        "task", "repeat", "outer_fold",
        "model", "selected_hyperparameter",
    ],
)
corrected_manifest = safe_read_csv(
    PATHS["goal2_corrected_manifest"],
    required=[
        "task", "repeat", "outer_fold",
        "selected_hyperparameter",
        "selected_residualizer_alpha",
    ],
)
authoritative_dx = safe_read_csv(
    PATHS["goal2_dx_oof"],
    required=[
        "participant_id", "logical_recording_id",
        "repeat", "outer_fold", "model",
        "y", "prediction", "row_weight",
    ],
)
authoritative_sev = safe_read_csv(
    PATHS["goal2_sev_oof"],
    required=[
        "participant_id", "logical_recording_id",
        "repeat", "outer_fold", "model",
        "y", "prediction", "row_weight",
    ],
)

for frame in [
    recording_table, participant_table, severity_pairs_raw, a_values,
    split_manifest, qchan_cache_index, authoritative_dx, authoritative_sev,
]:
    if "participant_id" in frame.columns:
        frame["participant_id"] = frame["participant_id"].astype(str).str.strip()
    if "logical_recording_id" in frame.columns:
        frame["logical_recording_id"] = (
            frame["logical_recording_id"].astype(str).str.strip()
        )
    if "assessment_date" in frame.columns:
        frame["assessment_date"] = frame["assessment_date"].astype(str)

a_manifest = json.loads(PATHS["a_manifest"].read_text(encoding="utf-8"))
if str(a_manifest.get("status", "")).upper() != "FROZEN":
    raise RuntimeError("A freeze manifest is not FROZEN.")
if sha256_file(PATHS["a_registry"]) != a_manifest["a_registry_sha256"]:
    raise RuntimeError("Frozen A registry hash mismatch.")
if sha256_file(PATHS["a_values"]) != a_manifest["acoustic_features_frozen_sha256"]:
    raise RuntimeError("Frozen A values hash mismatch.")
if int(a_manifest["n_primary_a"]) != EXPECTED["primary_A"]:
    raise RuntimeError("Primary-A count changed.")

# Canonical denominators.
recording_table["y_dx"] = recording_table["diagnosis"].map(
    {"ALS": 1, "CONTROLS": 0}
)
if recording_table["y_dx"].isna().any():
    raise ValueError("Unexpected diagnosis coding.")

assert len(recording_table) == EXPECTED["recordings"]
assert recording_table["participant_id"].nunique() == EXPECTED["participants"]
assert int((recording_table["y_dx"] == 1).sum()) == EXPECTED["als_recordings"]
assert int((recording_table["y_dx"] == 0).sum()) == EXPECTED["control_recordings"]

participant_dx = (
    recording_table[["participant_id", "y_dx"]]
    .drop_duplicates("participant_id")
)
assert len(participant_dx) == EXPECTED["participants"]
assert int(participant_dx["y_dx"].sum()) == EXPECTED["als_participants"]
assert int((participant_dx["y_dx"] == 0).sum()) == EXPECTED["control_participants"]

# Split contract.
assert len(split_manifest) == EXPECTED["participants"] * 10
assert split_manifest["repeat"].nunique() == 10
assert split_manifest["outer_fold"].nunique() == 5
assert split_manifest.groupby(["participant_id", "repeat"]).size().eq(1).all()

# Exact frozen feature definitions.
AGE = "age_at_recording_years"

PRIMARY_A = (
    a_registry.loc[a_registry["final_role"].eq("primary"), "feature"]
    .astype(str).tolist()
)
A_TRANSFORM = (
    a_registry.set_index("feature")["final_transform"].astype(str).to_dict()
)
A_SUPPORT = (
    a_registry.set_index("feature")["support_indicator_column"].astype(str).to_dict()
)

QADD = [
    "qadd_pause_ac_level_dbfs_median",
    "qadd_pause_level_iqr_db",
    "qadd_speech_pause_level_contrast_db",
]
QGAIN = [
    "qgain_typical_speech_level_dbfs",
    "qgain_within_segment_iqr_db",
    "qgain_between_segment_mad_db",
    "qgain_abs_drift_db_per_min",
]
QREV = ["qrev_srmr_norm"]

Q_TRANSFORM = dict(
    zip(
        q_registry["feature"].astype(str),
        q_registry["transform"].astype(str),
    )
)
Q_SUPPORT_LEGACY = [f"{feature}_supported" for feature in QADD]

# Pinned Paper-1 implementation.
if not (EXTERNAL / ".git").exists():
    raise FileNotFoundError(f"Pinned Paper 1 repository missing: {EXTERNAL}")

observed_paper1_commit = subprocess.check_output(
    ["git", "-C", str(EXTERNAL), "rev-parse", "HEAD"],
    text=True,
).strip()

if observed_paper1_commit != PAPER1_COMMIT:
    raise RuntimeError(
        "Paper 1 commit mismatch.\n"
        f"Expected: {PAPER1_COMMIT}\n"
        f"Observed: {observed_paper1_commit}"
    )

paper1_src = EXTERNAL / "src"
if str(paper1_src) not in sys.path:
    sys.path.insert(0, str(paper1_src))

from paper1_qc_reviewed.qchan_v400 import (
    ANALYSIS_FEATURES as QCHAN_FEATURES_TUPLE,
    DEFAULT_PARAMETERS as QCHAN_PARAMETERS,
    build_subject_balanced_loso_references,
    compute_reference_relative_features,
)
from paper1_qc_reviewed.qchan_v400_cohort import (
    load_recording_spectrum,
    save_reference_spectrum,
    load_reference_spectrum,
)

QCHAN = list(QCHAN_FEATURES_TUPLE)
CORE_Q = QADD + QGAIN + QREV + QCHAN
CORE_SUPPORT_SOURCES = list(QADD)

if len(CORE_Q) != 12:
    raise RuntimeError(f"Core-Q should contain 12 numeric features, found {len(CORE_Q)}.")
for feature in CORE_Q:
    if feature not in Q_TRANSFORM:
        raise KeyError(f"Core-Q transform missing from q_registry: {feature}")
for feature in PRIMARY_A:
    if feature not in a_values.columns:
        raise KeyError(f"Frozen Primary-A feature missing: {feature}")
    if A_SUPPORT[feature] not in a_values.columns:
        raise KeyError(f"Primary-A support indicator missing: {A_SUPPORT[feature]}")

# Load all 519 reference-independent QCHAN spectra.
spectra = {}
for row in qchan_cache_index.itertuples(index=False):
    cache_path = Path(str(row.cache_path))
    if not cache_path.is_absolute():
        cache_path = ROOT / cache_path
    if not cache_path.exists():
        raise FileNotFoundError(f"QCHAN spectrum cache missing: {cache_path}")
    spectrum = load_recording_spectrum(cache_path)
    if spectrum.status != "measured":
        raise RuntimeError(
            f"QCHAN spectrum not measured for {row.logical_recording_id}: {spectrum.status}"
        )
    spectra[str(row.logical_recording_id)] = spectrum

if len(spectra) != EXPECTED["recordings"]:
    raise RuntimeError(f"Expected 519 QCHAN spectra; loaded {len(spectra)}.")

print("AUTHORITATIVE INPUT / FEATURE CONTRACT GATE: PASS")
print("Primary-A:", PRIMARY_A)
print("Core-Q:", CORE_Q)
print("Paper 1 commit:", observed_paper1_commit)


AUTHORITATIVE INPUT / FEATURE CONTRACT GATE: PASS
Primary-A: ['bamboo_percent_pause_time_300ms', 'bamboo_pause_mean_sec_300ms', 'bamboo_phrase_mean_sec_300ms', 'bamboo_phrase_cv_300ms', 'bamboo_nominal_articulation_rate_syll_per_sec', 'bamboo_f0_iqr_semitones']
Core-Q: ['qadd_pause_ac_level_dbfs_median', 'qadd_pause_level_iqr_db', 'qadd_speech_pause_level_contrast_db', 'qgain_typical_speech_level_dbfs', 'qgain_within_segment_iqr_db', 'qgain_between_segment_mad_db', 'qgain_abs_drift_db_per_min', 'qrev_srmr_norm', 'qchan_ltas_distance_db', 'qchan_rolloff95_deficit_hz', 'qchan_highband_ratio_deficit', 'qchan_tilt_steepening_db_per_oct']
Paper 1 commit: cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8



## 3. Reconstruct the exact Goal 2 diagnosis and severity analysis populations

This deliberately uses the same row definitions as the authoritative Goal 2 completion:

- diagnosis: all retained recordings from age-complete participants;
- severity: all retained ALS recording–assessment pairs satisfying the frozen ≤60-day rule;
- participant-normalized fitting weights, so every participant contributes total fitting weight 1.


In [4]:

def coerce_bool_series(series, name):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    normalized = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": True, "1": True, "yes": True, "y": True,
        "false": False, "0": False, "no": False, "n": False,
    }
    unknown = sorted(set(normalized.dropna()) - set(mapping))
    if unknown:
        raise ValueError(f"{name}: unrecognized boolean values {unknown[:20]}")
    return normalized.map(mapping).astype(bool)

def participant_weights(frame):
    counts = frame.groupby("participant_id")["participant_id"].transform("size")
    weights = 1.0 / counts.to_numpy(float)
    check = (
        pd.DataFrame({
            "participant_id": frame["participant_id"].to_numpy(),
            "weight": weights,
        })
        .groupby("participant_id")["weight"]
        .sum()
    )
    if not np.allclose(check.to_numpy(float), 1.0, atol=1e-12, rtol=0):
        raise RuntimeError("Participant weights do not sum to one.")
    return weights

recording_model = recording_table.merge(
    a_values,
    on=["participant_id", "logical_recording_id"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_A"),
    indicator="_a_join",
)
if not recording_model["_a_join"].eq("both").all():
    raise RuntimeError("At least one canonical recording lacks frozen A.")
recording_model = recording_model.drop(columns=["_a_join"])

# Diagnosis primary population.
dx = recording_model.loc[recording_model[AGE].notna()].copy()
dx["y"] = dx["diagnosis"].map({"ALS": 1, "CONTROLS": 0})
if dx["y"].isna().any():
    raise ValueError("Unexpected diagnosis coding in diagnosis population.")

dx_participants = dx[["participant_id", "y"]].drop_duplicates("participant_id")
assert dx_participants["participant_id"].nunique() == EXPECTED["diagnosis_age_complete_participants"]
assert int(dx_participants["y"].sum()) == EXPECTED["diagnosis_age_complete_als"]
assert int((dx_participants["y"] == 0).sum()) == EXPECTED["diagnosis_age_complete_controls"]

# Severity primary population.
severity_pairs_raw["within_60_days"] = coerce_bool_series(
    severity_pairs_raw["within_60_days"], "within_60_days"
)
severity_pairs_60 = severity_pairs_raw.loc[
    severity_pairs_raw["within_60_days"]
].copy()

pair_fields = [
    "participant_id", "logical_recording_id",
    "bulbar_score", "abs_delta_days",
]
for optional in [
    "recording_date", "assessment_date",
    "alsfrs_total", "delta_days",
]:
    if optional in severity_pairs_60.columns:
        pair_fields.append(optional)

pair_meta = severity_pairs_60[pair_fields].copy()
if "assessment_date" in pair_meta.columns:
    pair_meta["assessment_date"] = pair_meta["assessment_date"].astype(str)

clinical_collision = {
    "bulbar_score", "alsfrs_total",
    "assessment_date", "delta_days",
    "abs_delta_days", "within_60_days", "within_90_days",
}
canonical_for_pair = recording_model.drop(
    columns=[c for c in clinical_collision if c in recording_model.columns],
    errors="ignore",
).copy()

if (
    "recording_date" in pair_meta.columns
    and "recording_date" in canonical_for_pair.columns
):
    canonical_for_pair = canonical_for_pair.drop(columns=["recording_date"])

sev = pair_meta.merge(
    canonical_for_pair,
    on=["participant_id", "logical_recording_id"],
    how="left",
    validate="many_to_one",
    indicator="_recording_join",
)
if len(sev) != len(pair_meta):
    raise RuntimeError("Severity enrichment changed pair-row count.")
if not sev["_recording_join"].eq("both").all():
    raise RuntimeError("Severity pair failed canonical+A join.")
sev = sev.drop(columns=["_recording_join"])

sev["y"] = pd.to_numeric(sev["bulbar_score"], errors="raise")
if not sev["y"].between(0, 12).all():
    raise ValueError("Bulbar score outside 0–12.")
if not sev["abs_delta_days"].le(60).all():
    raise RuntimeError("Primary severity contains >60-day pair.")
if sev[AGE].isna().any():
    raise RuntimeError("Primary severity contains missing age.")

assert len(sev) == EXPECTED["severity_pairs_60d"]
assert sev["participant_id"].nunique() == EXPECTED["severity_participants_60d"]

dx["row_weight"] = participant_weights(dx)
sev["row_weight"] = participant_weights(sev)

TASK_FRAMES = {
    "diagnosis": dx,
    "severity": sev,
}
AUTH_OOF = {
    "diagnosis": authoritative_dx,
    "severity": authoritative_sev,
}

population_summary = pd.DataFrame([
    {
        "task": "diagnosis",
        "rows": len(dx),
        "participants": dx["participant_id"].nunique(),
        "outcome": "ALS vs control",
    },
    {
        "task": "severity",
        "rows": len(sev),
        "participants": sev["participant_id"].nunique(),
        "outcome": "ALSFRS-R bulbar subscore",
    },
])

atomic_csv(population_summary, TABLES / "goal3_goal2_bridge_population_summary.csv")
display(population_summary)
print("EXACT GOAL 2 POPULATION RECONSTRUCTION: PASS")


,task,rows,participants,outcome
0,diagnosis,483,199,ALS vs control
1,severity,398,145,ALSFRS-R bulbar subscore


EXACT GOAL 2 POPULATION RECONSTRUCTION: PASS



## 4. Exact task/fold-specific QCHAN reconstruction and fixed external references

For model fitting, training recordings receive subject-level LOSO QCHAN references constructed only from the outer-training participant pool.

For later held-out Goal 3 inference, each task × fold receives one **fixed common external QCHAN reference** constructed exclusively from unmodified outer-training participants. That reference is serialized with the bridge and must be reused for baseline and all perturbed versions of the corresponding held-out source.

This is important: the diagnosis and severity training populations are not identical, so their fixed QCHAN references are stored separately.


In [5]:

def reference_metadata(reference_rows):
    meta = (
        reference_rows[["logical_recording_id", "participant_id"]]
        .drop_duplicates()
        .rename(columns={"participant_id": "subject_id"})
        .copy()
    )
    meta["logical_recording_id"] = meta["logical_recording_id"].astype(str)
    meta["subject_id"] = meta["subject_id"].astype(str)
    meta["task_stratum"] = "BAMBOO_PASSAGE"
    return meta

def build_qchan_context(reference_participant_ids, train_rows, target_rows, task, outer_fold):
    reference_participant_ids = set(map(str, reference_participant_ids))
    if not reference_participant_ids:
        raise ValueError("QCHAN reference participant set is empty.")

    reference_rows = (
        recording_table.loc[
            recording_table["participant_id"].isin(reference_participant_ids),
            ["participant_id", "logical_recording_id"],
        ]
        .drop_duplicates()
        .copy()
    )
    observed = set(reference_rows["participant_id"].astype(str))
    if observed != reference_participant_ids:
        missing = sorted(reference_participant_ids - observed)
        raise RuntimeError(
            f"{task}/fold{outer_fold}: training participants missing canonical recordings: "
            f"{missing[:10]}"
        )

    ref_spectra = {
        rid: spectra[rid]
        for rid in reference_rows["logical_recording_id"].astype(str)
    }
    ref_meta = reference_metadata(reference_rows)

    training_references = build_subject_balanced_loso_references(
        ref_spectra,
        ref_meta,
        parameters=QCHAN_PARAMETERS,
    )

    # Request a common outer-training-only reference using one external dummy target.
    target_unique = (
        target_rows[["participant_id", "logical_recording_id"]]
        .drop_duplicates()
        .copy()
    )
    if target_unique.empty:
        raise RuntimeError(f"{task}/fold{outer_fold}: no held-out target rows.")

    dummy = target_unique.iloc[0]
    dummy_rid = str(dummy["logical_recording_id"])
    dummy_pid = str(dummy["participant_id"])

    if dummy_pid in reference_participant_ids:
        raise RuntimeError("Chosen external QCHAN dummy is unexpectedly a training participant.")

    combo_spectra = dict(ref_spectra)
    combo_spectra[dummy_rid] = spectra[dummy_rid]
    combo_meta = pd.concat(
        [
            ref_meta,
            pd.DataFrame([{
                "logical_recording_id": dummy_rid,
                "subject_id": dummy_pid,
                "task_stratum": "BAMBOO_PASSAGE",
            }]),
        ],
        ignore_index=True,
    )

    refs = build_subject_balanced_loso_references(
        combo_spectra,
        combo_meta,
        parameters=QCHAN_PARAMETERS,
    )
    external_reference = refs[dummy_rid]

    members = set(map(str, external_reference.member_subject_ids))
    if not members.issubset(reference_participant_ids):
        raise RuntimeError(f"{task}/fold{outer_fold}: held-out QCHAN reference leakage.")
    if dummy_pid in members:
        raise RuntimeError(
            f"{task}/fold{outer_fold}: external dummy participant entered its own QCHAN reference."
        )
    heldout_ids = set(target_unique["participant_id"].astype(str))
    if members & heldout_ids:
        raise RuntimeError(
            f"{task}/fold{outer_fold}: fixed QCHAN reference contains held-out participants."
        )
    if external_reference.status != "measured":
        raise RuntimeError(
            f"{task}/fold{outer_fold}: external QCHAN reference is "
            f"{external_reference.status!r}, not 'measured'."
        )

    # Persist the official Paper-1 reference representation.
    ref_path = QCHAN_REFS / f"{task}_outer_fold_{outer_fold}_external_reference.npz"
    save_reference_spectrum(external_reference, ref_path)
    loaded_reference = load_reference_spectrum(ref_path)
    if loaded_reference.reference_sha256 != external_reference.reference_sha256:
        raise RuntimeError(
            f"{task}/fold{outer_fold}: QCHAN reference round-trip hash mismatch."
        )

    def compute_for(rows, *, training):
        unique = (
            rows[["participant_id", "logical_recording_id"]]
            .drop_duplicates()
            .copy()
        )
        output = []
        for r in unique.itertuples(index=False):
            pid = str(r.participant_id)
            rid = str(r.logical_recording_id)
            if training:
                reference = training_references[rid]
                train_members = set(map(str, reference.member_subject_ids))
                if pid in train_members:
                    raise RuntimeError(
                        f"{task}/fold{outer_fold}: QCHAN LOSO failure for {pid}."
                    )
                if not train_members.issubset(reference_participant_ids):
                    raise RuntimeError(
                        f"{task}/fold{outer_fold}: QCHAN training reference leakage."
                    )
            else:
                reference = external_reference

            values = compute_reference_relative_features(
                spectra[rid],
                reference,
                parameters=QCHAN_PARAMETERS,
            )
            output.append({
                "participant_id": pid,
                "logical_recording_id": rid,
                **{feature: values[feature] for feature in QCHAN},
            })

        out = pd.DataFrame(output)
        if len(out) != len(unique):
            raise RuntimeError("QCHAN output row-count mismatch.")
        if out[QCHAN].isna().any().any():
            raise RuntimeError("Fold-safe QCHAN produced missing values.")
        return out

    q_train = compute_for(train_rows, training=True)
    q_target = compute_for(target_rows, training=False)

    audit_row = {
        "task": task,
        "outer_repeat": OUTER_REPEAT,
        "outer_fold": int(outer_fold),
        "training_participants": len(reference_participant_ids),
        "heldout_participants": len(heldout_ids),
        "reference_recording_count": int(external_reference.recording_count),
        "reference_subject_count": int(external_reference.subject_count),
        "reference_sha256": str(external_reference.reference_sha256),
        "reference_file": str(ref_path.relative_to(ROOT)),
        "reference_file_sha256": sha256_file(ref_path),
        "heldout_subject_overlap_n": 0,
    }
    return q_train, q_target, audit_row

print("TASK/FOLD-SPECIFIC QCHAN BUILDER: READY")


TASK/FOLD-SPECIFIC QCHAN BUILDER: READY



## 5. Exact Goal 2 preprocessing and residualization semantics

The serialized bridge stores **state**, not notebook-local custom class instances. This avoids fragile pickle dependencies in Notebook 35.

For every fitted model the bundle contains:

- frozen A feature names/transforms;
- training-fold A medians and scaler;
- A support columns;
- frozen Q names/transforms when needed;
- training-fold Q medians/scaler;
- explicit QADD support semantics;
- corrected Q→A residualizer state for `M_A-resQ`;
- final design scaler;
- selected outcome hyperparameter;
- fitted scikit-learn outcome estimator;
- feature-column order;
- training/held-out participant IDs;
- path/hash of the fixed QCHAN reference for that task/fold.


In [6]:

def apply_transform(values, transform):
    x = pd.to_numeric(values, errors="coerce").to_numpy(float)
    transform = str(transform).strip().lower()

    if transform in {"none", "", "nan"}:
        return x
    if transform == "log1p":
        finite = np.isfinite(x)
        if finite.any() and np.nanmin(x[finite]) < 0:
            raise ValueError("log1p requested for negative values.")
        return np.log1p(x)
    if transform == "asinh":
        return np.arcsinh(x)
    raise ValueError(f"Unsupported frozen transform: {transform!r}")

def fit_numeric_state(frame, features, transform_map):
    features = list(features)
    columns = []
    for feature in features:
        if feature not in frame.columns:
            raise KeyError(f"Missing numeric predictor: {feature}")
        columns.append(apply_transform(frame[feature], transform_map[feature]))

    X = (
        np.column_stack(columns)
        if columns
        else np.empty((len(frame), 0), dtype=float)
    )
    medians = np.nanmedian(X, axis=0)

    if np.isnan(medians).any():
        bad = [
            feature for feature, median in zip(features, medians)
            if np.isnan(median)
        ]
        raise ValueError(
            "Training fold has no finite support for: " + ", ".join(bad)
        )

    X_imp = np.where(np.isnan(X), medians[None, :], X)
    scaler = StandardScaler().fit(X_imp)

    return {
        "features": features,
        "transform_map": {f: str(transform_map[f]) for f in features},
        "medians": np.asarray(medians, dtype=float),
        "scaler": scaler,
    }

def transform_numeric_state(frame, state):
    features = list(state["features"])
    columns = []
    for feature in features:
        if feature not in frame.columns:
            raise KeyError(f"Missing numeric predictor at inference: {feature}")
        columns.append(
            apply_transform(frame[feature], state["transform_map"][feature])
        )

    X = (
        np.column_stack(columns)
        if columns
        else np.empty((len(frame), 0), dtype=float)
    )
    medians = np.asarray(state["medians"], dtype=float)
    X_imp = np.where(np.isnan(X), medians[None, :], X)
    return state["scaler"].transform(X_imp)

def get_binary_matrix(frame, columns):
    columns = list(columns)
    if not columns:
        return np.empty((len(frame), 0), dtype=float)

    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError("Missing support predictor(s): " + ", ".join(missing))

    X = frame[columns].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    if not np.isfinite(X).all():
        raise ValueError("Support indicator contains missing/nonfinite values.")
    if not set(np.unique(X)).issubset({0.0, 1.0}):
        raise ValueError("Support indicator is not binary.")
    return X

def with_support_columns(frame, support_sources):
    out = frame.copy()
    cols = []
    for feature in support_sources:
        if feature not in out.columns:
            raise KeyError(f"Cannot create Q support indicator; missing: {feature}")
        col = f"__q_support__{feature}"
        out[col] = out[feature].notna().astype(float)
        cols.append(col)

        legacy = f"{feature}_supported"
        if legacy in out.columns:
            legacy_values = pd.to_numeric(
                out[legacy], errors="coerce"
            ).to_numpy(float)
            if not np.array_equal(
                legacy_values,
                out[col].to_numpy(float),
            ):
                raise RuntimeError(
                    f"Frozen support semantics changed for {feature}."
                )
    return out, cols

def fit_residualizer_state(
    Q_numeric,
    Q_support,
    A_numeric,
    A_support,
    sample_weight,
    alpha,
):
    q_raw = np.column_stack([Q_numeric, Q_support])
    q_scaler = StandardScaler()
    Q_design = q_scaler.fit_transform(q_raw)

    models = []
    for j in range(A_numeric.shape[1]):
        supported = A_support[:, j] == 1
        if supported.sum() < 20:
            raise RuntimeError(
                "Too few supported training rows to fit Q→A residualizer "
                f"for Primary-A column {j}: {supported.sum()}"
            )
        model = Ridge(alpha=float(alpha), fit_intercept=True)
        model.fit(
            Q_design[supported],
            A_numeric[supported, j],
            sample_weight=sample_weight[supported],
        )
        models.append(model)

    return {
        "alpha": float(alpha),
        "q_scaler": q_scaler,
        "models": models,
    }

def transform_residualizer_state(
    state,
    Q_numeric,
    Q_support,
    A_numeric,
    A_support,
):
    q_raw = np.column_stack([Q_numeric, Q_support])
    Q_design = state["q_scaler"].transform(q_raw)
    residual = np.asarray(A_numeric, dtype=float).copy()

    for j, model in enumerate(state["models"]):
        supported = A_support[:, j] == 1
        predicted = model.predict(Q_design)

        # Exact Goal-2 support rule:
        # residualize only genuinely observed A values.
        residual[supported, j] = (
            A_numeric[supported, j] - predicted[supported]
        )

        # Unsupported A retains the training-fold-imputed numeric placeholder.
        residual[~supported, j] = A_numeric[~supported, j]

    return residual

def make_spec(
    name,
    *,
    q_features=(),
    q_support_sources=(),
    q_binary_cols=(),
    include_age=True,
    include_sex=False,
    residualize=False,
):
    return {
        "name": str(name),
        "q_features": list(q_features),
        "q_support_sources": list(q_support_sources),
        "q_binary_cols": list(q_binary_cols),
        "include_age": bool(include_age),
        "include_sex": bool(include_sex),
        "residualize": bool(residualize),
    }

PRIMARY_SPECS = {
    "M_A": make_spec("M_A"),
    "M_A+Q": make_spec(
        "M_A+Q",
        q_features=CORE_Q,
        q_support_sources=CORE_SUPPORT_SOURCES,
    ),
    "M_A-resQ": make_spec(
        "M_A-resQ",
        q_features=CORE_Q,
        q_support_sources=CORE_SUPPORT_SOURCES,
        residualize=True,
    ),
}

def merge_qchan(frame, qchan_frame):
    out = frame.drop(
        columns=[c for c in QCHAN if c in frame.columns],
        errors="ignore",
    ).copy()
    out = out.merge(
        qchan_frame,
        on=["participant_id", "logical_recording_id"],
        how="left",
        validate="many_to_one",
    )
    if out[QCHAN].isna().any().any():
        raise RuntimeError("Fold-safe QCHAN merge produced missing values.")
    return out

print("EXACT PREPROCESSING / RESIDUALIZER STATE FUNCTIONS: READY")


EXACT PREPROCESSING / RESIDUALIZER STATE FUNCTIONS: READY



## 6. Fit one frozen model state and define the future inference interface

The same interface is used twice here:

1. immediately after fitting to reproduce the held-out Goal 2 prediction;
2. after `joblib` serialization/reload to verify that Notebook 35 will be able to reproduce the same prediction from the persisted state.


In [7]:

def fit_outcome_model(X, y, weights, task, hyperparameter):
    if task == "diagnosis":
        model = LogisticRegression(
            C=float(hyperparameter),
            solver="lbfgs",
            max_iter=5000,
            class_weight=None,
        )
        model.fit(
            X,
            y.astype(int),
            sample_weight=weights,
        )
    elif task == "severity":
        model = Ridge(alpha=float(hyperparameter))
        model.fit(
            X,
            y.astype(float),
            sample_weight=weights,
        )
    else:
        raise ValueError(task)
    return model

def predict_outcome(model, X, task):
    if task == "diagnosis":
        return model.predict_proba(X)[:, 1]
    if task == "severity":
        return model.predict(X)
    raise ValueError(task)

def fit_model_state(
    train,
    target,
    *,
    task,
    spec,
    outcome_hyperparameter,
    residualizer_alpha,
    q_train,
    q_target,
    qchan_reference_audit,
):
    train_local = train.copy()
    target_local = target.copy()

    q_features = list(spec["q_features"])
    needs_qchan = any(f in QCHAN for f in q_features)
    if needs_qchan:
        train_local = merge_qchan(train_local, q_train)
        target_local = merge_qchan(target_local, q_target)

    # A: same frozen Primary-A in every representation.
    a_state = fit_numeric_state(
        train_local,
        PRIMARY_A,
        {f: A_TRANSFORM[f] for f in PRIMARY_A},
    )
    A_train = transform_numeric_state(train_local, a_state)
    A_target = transform_numeric_state(target_local, a_state)

    a_support_cols = [A_SUPPORT[f] for f in PRIMARY_A]
    A_sup_train = get_binary_matrix(train_local, a_support_cols)
    A_sup_target = get_binary_matrix(target_local, a_support_cols)

    # Q: explicit support handling exactly as Goal 2.
    if q_features:
        train_local, q_support_cols = with_support_columns(
            train_local, spec["q_support_sources"]
        )
        target_local, q_support_cols_target = with_support_columns(
            target_local, spec["q_support_sources"]
        )
        if q_support_cols != q_support_cols_target:
            raise RuntimeError("Q support column mismatch train/target.")

        q_state = fit_numeric_state(
            train_local,
            q_features,
            {f: Q_TRANSFORM[f] for f in q_features},
        )
        Q_train = transform_numeric_state(train_local, q_state)
        Q_target = transform_numeric_state(target_local, q_state)

        Q_sup_train = get_binary_matrix(train_local, q_support_cols)
        Q_sup_target = get_binary_matrix(target_local, q_support_cols)

        binary_cols = list(spec["q_binary_cols"])
        Q_bin_train = get_binary_matrix(train_local, binary_cols)
        Q_bin_target = get_binary_matrix(target_local, binary_cols)

        Q_aux_train = np.column_stack([Q_sup_train, Q_bin_train])
        Q_aux_target = np.column_stack([Q_sup_target, Q_bin_target])
    else:
        q_state = None
        Q_train = np.empty((len(train_local), 0), float)
        Q_target = np.empty((len(target_local), 0), float)
        Q_aux_train = np.empty((len(train_local), 0), float)
        Q_aux_target = np.empty((len(target_local), 0), float)
        q_support_cols = []
        binary_cols = []

    # Covariate block: age only in the frozen primary specifications.
    cov_train, cov_target, cov_names = [], [], []

    if spec["include_age"]:
        age_train = pd.to_numeric(
            train_local[AGE], errors="coerce"
        ).to_numpy(float)[:, None]
        age_target = pd.to_numeric(
            target_local[AGE], errors="coerce"
        ).to_numpy(float)[:, None]
        if not np.isfinite(age_train).all() or not np.isfinite(age_target).all():
            raise ValueError(
                f"{spec['name']}: age missing in complete-case analysis."
            )
        cov_train.append(age_train)
        cov_target.append(age_target)
        cov_names.append("Age")

    if spec["include_sex"]:
        raise RuntimeError("Primary Goal-2 bridge must not unexpectedly include sex.")

    C_train = (
        np.column_stack(cov_train)
        if cov_train else np.empty((len(train_local), 0), float)
    )
    C_target = (
        np.column_stack(cov_target)
        if cov_target else np.empty((len(target_local), 0), float)
    )

    residualizer_state = None
    if spec["residualize"]:
        if residualizer_alpha is None:
            raise ValueError("M_A-resQ requires corrected selected residualizer alpha.")
        residualizer_state = fit_residualizer_state(
            Q_train,
            Q_aux_train,
            A_train,
            A_sup_train,
            train_local["row_weight"].to_numpy(float),
            residualizer_alpha,
        )
        A_used_train = transform_residualizer_state(
            residualizer_state,
            Q_train, Q_aux_train,
            A_train, A_sup_train,
        )
        A_used_target = transform_residualizer_state(
            residualizer_state,
            Q_target, Q_aux_target,
            A_target, A_sup_target,
        )
        a_names = [f"{f}_resQ" for f in PRIMARY_A]
    else:
        A_used_train = A_train
        A_used_target = A_target
        a_names = list(PRIMARY_A)

    raw_train_parts = [C_train, A_used_train, A_sup_train]
    raw_target_parts = [C_target, A_used_target, A_sup_target]
    feature_names = (
        cov_names
        + a_names
        + [f"support:{c}" for c in a_support_cols]
    )

    # Q enters the clinical design only for M_A+Q.
    if q_features and not spec["residualize"]:
        raw_train_parts.extend([Q_train, Q_aux_train])
        raw_target_parts.extend([Q_target, Q_aux_target])
        feature_names.extend(q_features)
        feature_names.extend([f"support:{c}" for c in q_support_cols])
        feature_names.extend([f"binary:{c}" for c in binary_cols])

    raw_train = np.column_stack(raw_train_parts)
    raw_target = np.column_stack(raw_target_parts)

    final_scaler = StandardScaler()
    X_train = final_scaler.fit_transform(raw_train)
    X_target = final_scaler.transform(raw_target)

    if not np.isfinite(X_train).all() or not np.isfinite(X_target).all():
        raise RuntimeError(
            f"{task}/{spec['name']}: nonfinite design after preprocessing."
        )

    outcome_model = fit_outcome_model(
        X_train,
        train_local["y"].to_numpy(float),
        train_local["row_weight"].to_numpy(float),
        task,
        outcome_hyperparameter,
    )
    prediction = predict_outcome(outcome_model, X_target, task)

    state = {
        "schema_version": "goal3-goal2-model-state-v1.0",
        "task": task,
        "model": spec["name"],
        "outer_repeat": OUTER_REPEAT,
        "spec": dict(spec),
        "primary_A": list(PRIMARY_A),
        "core_Q": list(CORE_Q),
        "a_support_cols": list(a_support_cols),
        "q_support_cols": list(q_support_cols),
        "q_binary_cols": list(binary_cols),
        "a_preprocessor": a_state,
        "q_preprocessor": q_state,
        "residualizer": residualizer_state,
        "final_scaler": final_scaler,
        "outcome_model": outcome_model,
        "selected_outcome_hyperparameter": float(outcome_hyperparameter),
        "selected_residualizer_alpha": (
            None if residualizer_alpha is None
            else float(residualizer_alpha)
        ),
        "feature_names": list(feature_names),
        "qchan_reference_file": qchan_reference_audit["reference_file"],
        "qchan_reference_sha256": qchan_reference_audit["reference_sha256"],
        "qchan_reference_file_sha256": qchan_reference_audit["reference_file_sha256"],
        "paper1_commit": observed_paper1_commit,
        "A_freeze_version": str(a_manifest["a_freeze_version"]),
        "training_participant_ids": sorted(
            train_local["participant_id"].astype(str).unique().tolist()
        ),
        "heldout_participant_ids": sorted(
            target_local["participant_id"].astype(str).unique().tolist()
        ),
    }

    return state, prediction

def inference_design_from_state(target, state, q_target):
    """
    Exact persisted-state inference path that Notebook 35 must use after
    it has freshly remeasured Q and A on a baseline/perturbed held-out waveform.
    """
    local = target.copy()
    spec = state["spec"]
    q_features = list(spec["q_features"])

    if q_features and any(f in QCHAN for f in q_features):
        local = merge_qchan(local, q_target)

    A = transform_numeric_state(local, state["a_preprocessor"])
    A_sup = get_binary_matrix(local, state["a_support_cols"])

    if q_features:
        local, q_support_cols = with_support_columns(
            local, spec["q_support_sources"]
        )
        if q_support_cols != state["q_support_cols"]:
            raise RuntimeError("Persisted Q support-column contract changed.")

        Q = transform_numeric_state(local, state["q_preprocessor"])
        Q_sup = get_binary_matrix(local, q_support_cols)
        Q_bin = get_binary_matrix(local, state["q_binary_cols"])
        Q_aux = np.column_stack([Q_sup, Q_bin])
    else:
        Q = np.empty((len(local), 0), float)
        Q_aux = np.empty((len(local), 0), float)

    cov = []
    if spec["include_age"]:
        age = pd.to_numeric(local[AGE], errors="coerce").to_numpy(float)[:, None]
        if not np.isfinite(age).all():
            raise ValueError("Age missing at bundle inference.")
        cov.append(age)
    C = np.column_stack(cov) if cov else np.empty((len(local), 0), float)

    if spec["residualize"]:
        A_used = transform_residualizer_state(
            state["residualizer"],
            Q, Q_aux, A, A_sup,
        )
    else:
        A_used = A

    raw_parts = [C, A_used, A_sup]
    if q_features and not spec["residualize"]:
        raw_parts.extend([Q, Q_aux])

    raw = np.column_stack(raw_parts)
    X = state["final_scaler"].transform(raw)

    if X.shape[1] != len(state["feature_names"]):
        raise RuntimeError(
            f"Persisted design width changed: {X.shape[1]} vs "
            f"{len(state['feature_names'])}."
        )
    if not np.isfinite(X).all():
        raise RuntimeError("Persisted inference design contains nonfinite values.")

    return X

def predict_from_state(target, state, q_target):
    X = inference_design_from_state(target, state, q_target)
    return predict_outcome(state["outcome_model"], X, state["task"])

print("FROZEN MODEL-STATE FIT / INFERENCE INTERFACE: READY")


FROZEN MODEL-STATE FIT / INFERENCE INTERFACE: READY



## 7. Build the five outer-fold bundle files and reproduce authoritative repeat-1 OOF predictions

For `M_A` and `M_A+Q`, the selected outcome hyperparameters are read from the original sealed Goal 2 primary fold manifest.

For the authoritative corrected `M_A-resQ`, both the selected outcome hyperparameter and the selected Q→A residualizer alpha are read from the Goal 2 completion manifest.

No inner tuning is repeated here. This bridge reconstructs the already selected authoritative models exactly; it does not reselect them.

**Acceptance criterion:** every held-out repeat-1 prediction must reproduce the authoritative Goal 2 OOF prediction within `atol=rtol=1e-10`, including after bundle serialization and reload.


In [8]:

def lookup_hyperparameters(task, outer_fold, model_name):
    if model_name == "M_A-resQ":
        row = corrected_manifest.loc[
            corrected_manifest["task"].astype(str).eq(task)
            & corrected_manifest["repeat"].astype(int).eq(OUTER_REPEAT)
            & corrected_manifest["outer_fold"].astype(int).eq(int(outer_fold))
        ].copy()
        if len(row) != 1:
            raise RuntimeError(
                f"Corrected M_A-resQ manifest lookup failed for "
                f"{task}/fold{outer_fold}: {len(row)} rows."
            )
        return (
            float(row["selected_hyperparameter"].iloc[0]),
            float(row["selected_residualizer_alpha"].iloc[0]),
        )

    row = primary_manifest.loc[
        primary_manifest["task"].astype(str).eq(task)
        & primary_manifest["repeat"].astype(int).eq(OUTER_REPEAT)
        & primary_manifest["outer_fold"].astype(int).eq(int(outer_fold))
        & primary_manifest["model"].astype(str).eq(model_name)
    ].copy()
    if len(row) != 1:
        raise RuntimeError(
            f"Primary manifest lookup failed for "
            f"{task}/fold{outer_fold}/{model_name}: {len(row)} rows."
        )
    return float(row["selected_hyperparameter"].iloc[0]), None

def comparison_identity(task, frame_left, frame_right):
    keys = ["participant_id", "logical_recording_id"]
    if (
        task == "severity"
        and "assessment_date" in frame_left.columns
        and "assessment_date" in frame_right.columns
    ):
        keys.append("assessment_date")
    return keys

reproduction_rows = []
summary_rows = []
bundle_manifest_rows = []
qchan_audit_rows = []
participant_rows = []

for outer_fold in range(1, OUTER_FOLDS + 1):
    print("=" * 76)
    print(f"BUILDING OUTER FOLD {outer_fold}/{OUTER_FOLDS}")
    print("=" * 76)

    fold_bundle = {
        "schema_version": "goal3-goal2-outer-fold-bundle-v1.0",
        "engine_version": ENGINE_VERSION,
        "run_signature": RUN_SIGNATURE,
        "paper1_commit": observed_paper1_commit,
        "outer_repeat": OUTER_REPEAT,
        "outer_fold": outer_fold,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "tasks": {},
    }

    # Keep QCHAN frames temporarily for the post-serialization round trip.
    roundtrip_context = {}

    for task, frame in TASK_FRAMES.items():
        manifest_slice = split_manifest.loc[
            split_manifest["repeat"].astype(int).eq(OUTER_REPEAT)
            & split_manifest["outer_fold"].astype(int).eq(outer_fold)
        ]
        heldout_ids = set(manifest_slice["participant_id"].astype(str))
        heldout_ids &= set(frame["participant_id"].astype(str))

        train = frame.loc[
            ~frame["participant_id"].isin(heldout_ids)
        ].copy()
        target = frame.loc[
            frame["participant_id"].isin(heldout_ids)
        ].copy()

        if train.empty or target.empty:
            raise RuntimeError(f"{task}/fold{outer_fold}: empty train/test split.")
        if set(train["participant_id"]) & set(target["participant_id"]):
            raise RuntimeError(f"{task}/fold{outer_fold}: participant leakage.")

        # Recompute total fitting weight within the actual outer-training split.
        train["row_weight"] = participant_weights(train)
        target["row_weight"] = participant_weights(target)

        for pid in sorted(train["participant_id"].astype(str).unique()):
            participant_rows.append({
                "task": task,
                "outer_repeat": OUTER_REPEAT,
                "outer_fold": outer_fold,
                "role": "train",
                "participant_id": pid,
            })
        for pid in sorted(target["participant_id"].astype(str).unique()):
            participant_rows.append({
                "task": task,
                "outer_repeat": OUTER_REPEAT,
                "outer_fold": outer_fold,
                "role": "heldout",
                "participant_id": pid,
            })

        # Exact training-only QCHAN state for this task/fold.
        q_train, q_target, qchan_audit = build_qchan_context(
            set(train["participant_id"].astype(str)),
            train,
            target,
            task,
            outer_fold,
        )
        qchan_audit_rows.append(qchan_audit)

        fold_bundle["tasks"][task] = {
            "training_participant_ids": sorted(
                train["participant_id"].astype(str).unique().tolist()
            ),
            "heldout_participant_ids": sorted(
                target["participant_id"].astype(str).unique().tolist()
            ),
            "qchan_reference": qchan_audit,
            "models": {},
        }
        roundtrip_context[task] = {
            "target": target.copy(),
            "q_target": q_target.copy(),
        }

        authoritative = AUTH_OOF[task].loc[
            AUTH_OOF[task]["repeat"].astype(int).eq(OUTER_REPEAT)
            & AUTH_OOF[task]["outer_fold"].astype(int).eq(outer_fold)
        ].copy()

        for model_name in MODELS:
            outcome_hp, resid_alpha = lookup_hyperparameters(
                task, outer_fold, model_name
            )

            state, pred = fit_model_state(
                train,
                target,
                task=task,
                spec=PRIMARY_SPECS[model_name],
                outcome_hyperparameter=outcome_hp,
                residualizer_alpha=resid_alpha,
                q_train=q_train,
                q_target=q_target,
                qchan_reference_audit=qchan_audit,
            )
            state["outer_fold"] = outer_fold

            fold_bundle["tasks"][task]["models"][model_name] = state

            keys = comparison_identity(task, target, authoritative)
            local = target[keys + ["y"]].copy()
            local["reconstructed_prediction"] = np.asarray(pred, dtype=float)

            auth = authoritative.loc[
                authoritative["model"].astype(str).eq(model_name),
                keys + ["y", "prediction"],
            ].copy().rename(
                columns={
                    "y": "authoritative_y",
                    "prediction": "authoritative_prediction",
                }
            )

            compare = local.merge(
                auth,
                on=keys,
                how="inner",
                validate="one_to_one",
            )
            if len(compare) != len(local) or len(compare) != len(auth):
                raise RuntimeError(
                    f"{task}/fold{outer_fold}/{model_name}: authoritative "
                    "OOF identity coverage mismatch."
                )

            if not np.allclose(
                compare["y"].to_numpy(float),
                compare["authoritative_y"].to_numpy(float),
                atol=0,
                rtol=0,
            ):
                raise RuntimeError(
                    f"{task}/fold{outer_fold}/{model_name}: outcome mismatch."
                )

            compare["abs_diff_in_memory"] = np.abs(
                compare["reconstructed_prediction"].to_numpy(float)
                - compare["authoritative_prediction"].to_numpy(float)
            )
            compare["pass_in_memory"] = np.isclose(
                compare["reconstructed_prediction"].to_numpy(float),
                compare["authoritative_prediction"].to_numpy(float),
                atol=PRED_ATOL,
                rtol=PRED_RTOL,
            )

            if not compare["pass_in_memory"].all():
                bad = compare.loc[~compare["pass_in_memory"]].sort_values(
                    "abs_diff_in_memory", ascending=False
                )
                display(bad.head(20))
                raise RuntimeError(
                    f"{task}/fold{outer_fold}/{model_name}: in-memory "
                    "Goal-2 OOF reproduction FAILED."
                )

            # Save observation-level comparison now; round-trip fields added below.
            compare["task"] = task
            compare["outer_repeat"] = OUTER_REPEAT
            compare["outer_fold"] = outer_fold
            compare["model"] = model_name
            reproduction_rows.append(compare)

            bundle_manifest_rows.append({
                "task": task,
                "outer_repeat": OUTER_REPEAT,
                "outer_fold": outer_fold,
                "model": model_name,
                "selected_outcome_hyperparameter": outcome_hp,
                "selected_residualizer_alpha": resid_alpha,
                "n_train_rows": len(train),
                "n_train_participants": train["participant_id"].nunique(),
                "n_heldout_rows": len(target),
                "n_heldout_participants": target["participant_id"].nunique(),
                "design_columns": len(state["feature_names"]),
                "qchan_reference_sha256": qchan_audit["reference_sha256"],
                "qchan_reference_file": qchan_audit["reference_file"],
            })

            print(
                f"  {task:9s} | {model_name:8s} | "
                f"in-memory max |Δ| = {compare['abs_diff_in_memory'].max():.3e}"
            )

    # One persisted bundle per outer fold, containing both tasks and all 3 models.
    bundle_path = BUNDLES / f"goal2_outer_repeat1_fold_{outer_fold}.joblib"
    atomic_joblib(fold_bundle, bundle_path)

    # Mandatory serialization/reload reproduction gate.
    loaded = joblib.load(bundle_path)
    if loaded["run_signature"] != RUN_SIGNATURE:
        raise RuntimeError(f"Fold {outer_fold}: loaded bundle signature mismatch.")

    for task in ["diagnosis", "severity"]:
        target = roundtrip_context[task]["target"]
        q_target = roundtrip_context[task]["q_target"]
        authoritative = AUTH_OOF[task].loc[
            AUTH_OOF[task]["repeat"].astype(int).eq(OUTER_REPEAT)
            & AUTH_OOF[task]["outer_fold"].astype(int).eq(outer_fold)
        ].copy()

        for model_name in MODELS:
            state = loaded["tasks"][task]["models"][model_name]
            pred_loaded = predict_from_state(target, state, q_target)

            keys = comparison_identity(task, target, authoritative)
            local = target[keys].copy()
            local["roundtrip_prediction"] = np.asarray(pred_loaded, dtype=float)

            auth = authoritative.loc[
                authoritative["model"].astype(str).eq(model_name),
                keys + ["prediction"],
            ].copy().rename(
                columns={"prediction": "authoritative_prediction"}
            )

            rt = local.merge(
                auth,
                on=keys,
                how="inner",
                validate="one_to_one",
            )
            if len(rt) != len(local) or len(rt) != len(auth):
                raise RuntimeError(
                    f"{task}/fold{outer_fold}/{model_name}: bundle round-trip "
                    "identity coverage mismatch."
                )

            rt["abs_diff_roundtrip"] = np.abs(
                rt["roundtrip_prediction"].to_numpy(float)
                - rt["authoritative_prediction"].to_numpy(float)
            )
            rt["pass_roundtrip"] = np.isclose(
                rt["roundtrip_prediction"].to_numpy(float),
                rt["authoritative_prediction"].to_numpy(float),
                atol=PRED_ATOL,
                rtol=PRED_RTOL,
            )

            if not rt["pass_roundtrip"].all():
                display(
                    rt.loc[~rt["pass_roundtrip"]]
                    .sort_values("abs_diff_roundtrip", ascending=False)
                    .head(20)
                )
                raise RuntimeError(
                    f"{task}/fold{outer_fold}/{model_name}: serialized bundle "
                    "OOF reproduction FAILED."
                )

            # Attach round-trip result to the already-created observation table.
            idx = next(
                i for i in range(len(reproduction_rows) - 1, -1, -1)
                if reproduction_rows[i]["task"].iloc[0] == task
                and int(reproduction_rows[i]["outer_fold"].iloc[0]) == outer_fold
                and reproduction_rows[i]["model"].iloc[0] == model_name
            )
            current = reproduction_rows[idx]
            rt_small = rt[keys + ["roundtrip_prediction", "abs_diff_roundtrip", "pass_roundtrip"]]
            current = current.merge(
                rt_small,
                on=keys,
                how="left",
                validate="one_to_one",
            )
            reproduction_rows[idx] = current

            summary_rows.append({
                "task": task,
                "outer_repeat": OUTER_REPEAT,
                "outer_fold": outer_fold,
                "model": model_name,
                "n_predictions": len(rt),
                "max_abs_diff_in_memory": float(
                    reproduction_rows[idx]["abs_diff_in_memory"].max()
                ),
                "max_abs_diff_roundtrip": float(rt["abs_diff_roundtrip"].max()),
                "mean_abs_diff_roundtrip": float(rt["abs_diff_roundtrip"].mean()),
                "all_in_memory_pass": bool(
                    reproduction_rows[idx]["pass_in_memory"].all()
                ),
                "all_roundtrip_pass": bool(rt["pass_roundtrip"].all()),
                "selected_outcome_hyperparameter": float(
                    loaded["tasks"][task]["models"][model_name][
                        "selected_outcome_hyperparameter"
                    ]
                ),
                "selected_residualizer_alpha": loaded["tasks"][task]["models"][
                    model_name
                ]["selected_residualizer_alpha"],
                "bundle_file": str(bundle_path.relative_to(ROOT)),
                "bundle_file_sha256": sha256_file(bundle_path),
            })

    print(
        f"FOLD {outer_fold}: SERIALIZED BUNDLE REPRODUCTION PASS | "
        f"{bundle_path.name}"
    )

print("=" * 76)
print("ALL FIVE OUTER-FOLD BUNDLES BUILT AND REPRODUCED")
print("=" * 76)


BUILDING OUTER FOLD 1/5
  diagnosis | M_A      | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A+Q    | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A-resQ | in-memory max |Δ| = 1.110e-16
  severity  | M_A      | in-memory max |Δ| = 1.776e-15
  severity  | M_A+Q    | in-memory max |Δ| = 1.776e-15
  severity  | M_A-resQ | in-memory max |Δ| = 1.776e-15
FOLD 1: SERIALIZED BUNDLE REPRODUCTION PASS | goal2_outer_repeat1_fold_1.joblib
BUILDING OUTER FOLD 2/5


C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


  diagnosis | M_A      | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A+Q    | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A-resQ | in-memory max |Δ| = 1.110e-16
  severity  | M_A      | in-memory max |Δ| = 1.776e-15
  severity  | M_A+Q    | in-memory max |Δ| = 1.776e-15
  severity  | M_A-resQ | in-memory max |Δ| = 1.776e-15
FOLD 2: SERIALIZED BUNDLE REPRODUCTION PASS | goal2_outer_repeat1_fold_2.joblib
BUILDING OUTER FOLD 3/5


C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


  diagnosis | M_A      | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A+Q    | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A-resQ | in-memory max |Δ| = 1.110e-16
  severity  | M_A      | in-memory max |Δ| = 1.776e-15
  severity  | M_A+Q    | in-memory max |Δ| = 1.776e-15
  severity  | M_A-resQ | in-memory max |Δ| = 1.776e-15
FOLD 3: SERIALIZED BUNDLE REPRODUCTION PASS | goal2_outer_repeat1_fold_3.joblib
BUILDING OUTER FOLD 4/5


C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


  diagnosis | M_A      | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A+Q    | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A-resQ | in-memory max |Δ| = 1.110e-16
  severity  | M_A      | in-memory max |Δ| = 1.776e-15
  severity  | M_A+Q    | in-memory max |Δ| = 1.776e-15
  severity  | M_A-resQ | in-memory max |Δ| = 1.776e-15
FOLD 4: SERIALIZED BUNDLE REPRODUCTION PASS | goal2_outer_repeat1_fold_4.joblib
BUILDING OUTER FOLD 5/5


C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


  diagnosis | M_A      | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A+Q    | in-memory max |Δ| = 1.110e-16
  diagnosis | M_A-resQ | in-memory max |Δ| = 1.110e-16
  severity  | M_A      | in-memory max |Δ| = 1.776e-15
  severity  | M_A+Q    | in-memory max |Δ| = 1.776e-15
  severity  | M_A-resQ | in-memory max |Δ| = 1.776e-15
FOLD 5: SERIALIZED BUNDLE REPRODUCTION PASS | goal2_outer_repeat1_fold_5.joblib
ALL FIVE OUTER-FOLD BUNDLES BUILT AND REPRODUCED


C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape



## 8. Seal the Goal 2 → Goal 3 model bridge

A PASS seal is written only if:

- there are exactly 30 task × fold × model reconstruction summaries;
- every in-memory comparison passes;
- every post-serialization comparison passes;
- all five outer-fold bundle files exist and are nonempty;
- all ten task × fold fixed QCHAN reference files exist and are nonempty;
- participant train/held-out membership is disjoint within every task/fold;
- all exact output artifacts are hashed.

Only after this cell prints `GOAL 3 GOAL-2 MODEL BUNDLE BRIDGE: PASS` is the full controlled perturbation notebook allowed to run.


In [9]:

reproduction = pd.concat(reproduction_rows, ignore_index=True, sort=False)
reproduction_summary = pd.DataFrame(summary_rows)
bundle_manifest = pd.DataFrame(bundle_manifest_rows)
qchan_audit = pd.DataFrame(qchan_audit_rows)
fold_participants = pd.DataFrame(participant_rows)

if len(reproduction_summary) != 2 * OUTER_FOLDS * len(MODELS):
    raise RuntimeError(
        f"Expected 30 reproduction summary rows; found {len(reproduction_summary)}."
    )
if not reproduction_summary["all_in_memory_pass"].all():
    raise RuntimeError("At least one in-memory Goal-2 reproduction failed.")
if not reproduction_summary["all_roundtrip_pass"].all():
    raise RuntimeError("At least one serialized bundle reproduction failed.")
if reproduction_summary["max_abs_diff_roundtrip"].max() > (
    PRED_ATOL + PRED_RTOL * max(
        1.0,
        float(np.nanmax(np.abs(reproduction["authoritative_prediction"]))),
    )
):
    # np.isclose above is the authoritative per-row rule; this is a coarse
    # additional guard against an unexpectedly large reported maximum.
    raise RuntimeError("Unexpectedly large bundle reproduction discrepancy.")

# Disjoint participant membership within task/fold.
for (task, fold), local in fold_participants.groupby(["task", "outer_fold"]):
    train_ids = set(local.loc[local["role"].eq("train"), "participant_id"])
    test_ids = set(local.loc[local["role"].eq("heldout"), "participant_id"])
    if train_ids & test_ids:
        raise RuntimeError(f"{task}/fold{fold}: participant leakage in saved membership.")

# Exact file-count gates.
bundle_files = sorted(BUNDLES.glob("goal2_outer_repeat1_fold_*.joblib"))
qchan_files = sorted(QCHAN_REFS.glob("*_external_reference.npz"))

if len(bundle_files) != OUTER_FOLDS:
    raise RuntimeError(f"Expected 5 bundle files; found {len(bundle_files)}.")
if len(qchan_files) != 2 * OUTER_FOLDS:
    raise RuntimeError(f"Expected 10 task/fold QCHAN references; found {len(qchan_files)}.")

for path in bundle_files + qchan_files:
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Missing/empty final bridge artifact: {path}")

# Save exact audit/source tables.
atomic_csv(
    reproduction,
    TABLES / "goal3_goal2_repeat1_prediction_reproduction.csv",
)
atomic_csv(
    reproduction_summary,
    TABLES / "goal3_goal2_reproduction_summary.csv",
)
atomic_csv(
    bundle_manifest,
    TABLES / "goal3_goal2_model_bundle_manifest.csv",
)
atomic_csv(
    qchan_audit,
    TABLES / "goal3_goal2_qchan_reference_audit.csv",
)
atomic_csv(
    fold_participants,
    TABLES / "goal3_goal2_fold_participants.csv",
)

required_outputs = [
    TABLES / "goal3_goal2_repeat1_prediction_reproduction.csv",
    TABLES / "goal3_goal2_reproduction_summary.csv",
    TABLES / "goal3_goal2_model_bundle_manifest.csv",
    TABLES / "goal3_goal2_qchan_reference_audit.csv",
    TABLES / "goal3_goal2_fold_participants.csv",
    AUDIT / "goal3_goal2_bridge_input_hashes.json",
    OUT / "RUN_SIGNATURE.json",
    *bundle_files,
    *qchan_files,
]

artifact_hashes = {
    str(path.relative_to(ROOT)): sha256_file(path)
    for path in required_outputs
}

seal = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "goal": 3,
    "stage": "Goal2_to_Goal3_model_bundle_bridge",
    "engine_version": ENGINE_VERSION,
    "run_signature": RUN_SIGNATURE,
    "outer_repeat": OUTER_REPEAT,
    "outer_folds": OUTER_FOLDS,
    "tasks": ["diagnosis", "severity"],
    "models": MODELS,
    "bundle_count": len(bundle_files),
    "model_state_count": len(reproduction_summary),
    "qchan_reference_count": len(qchan_files),
    "prediction_reproduction_atol": PRED_ATOL,
    "prediction_reproduction_rtol": PRED_RTOL,
    "maximum_absolute_roundtrip_difference": float(
        reproduction_summary["max_abs_diff_roundtrip"].max()
    ),
    "paper1_commit": observed_paper1_commit,
    "A_freeze_version": str(a_manifest["a_freeze_version"]),
    "perturbation_manifest_sha256": input_hashes["goal3_perturbation_manifest"],
    "interpretation_boundary": (
        "This seal establishes faithful reconstruction and persistence of the "
        "authoritative first-repeat Goal 2 fold-specific clinical models. "
        "No controlled perturbation outcome has been evaluated in this stage."
    ),
    "next_allowed_step": (
        "Execute the full controlled Goal 3 perturbation experiment using only "
        "these sealed fold/task model bundles and their fixed QCHAN references. "
        "Perturbed audio must never retrain or refit any bundle component."
    ),
    "artifact_hashes": artifact_hashes,
}

atomic_json(seal, OUT / "GOAL3_GOAL2_MODEL_BUNDLE_SEAL.json")
atomic_json(seal, OUT / "DONE.json")

display(
    reproduction_summary[
        [
            "task", "outer_fold", "model", "n_predictions",
            "selected_outcome_hyperparameter",
            "selected_residualizer_alpha",
            "max_abs_diff_in_memory",
            "max_abs_diff_roundtrip",
            "all_roundtrip_pass",
        ]
    ].sort_values(["task", "outer_fold", "model"])
)

display(qchan_audit.sort_values(["task", "outer_fold"]))

print("=" * 76)
print("GOAL 3 GOAL-2 MODEL BUNDLE BRIDGE: PASS")
print("=" * 76)
print("Seal:", OUT / "GOAL3_GOAL2_MODEL_BUNDLE_SEAL.json")
print("Five bundle files:")
for path in bundle_files:
    print("  ", path)
print()
print("Maximum absolute post-serialization prediction difference:")
print(f"  {seal['maximum_absolute_roundtrip_difference']:.3e}")
print()
print("NEXT ALLOWED STEP:")
print(seal["next_allowed_step"])


,task,outer_fold,model,n_predictions,selected_outcome_hyperparameter,selected_residualizer_alpha,max_abs_diff_in_memory,max_abs_diff_roundtrip,all_roundtrip_pass
0,diagnosis,1,M_A,98,0.1000,NaN,1.110223e-16,1.110223e-16,True
1,diagnosis,1,M_A+Q,98,0.1000,NaN,1.110223e-16,1.110223e-16,True
2,diagnosis,1,M_A-resQ,98,0.1000,10.0,1.110223e-16,1.110223e-16,True
6,diagnosis,2,M_A,101,0.1000,NaN,1.110223e-16,1.110223e-16,True
7,diagnosis,2,M_A+Q,101,0.1000,NaN,1.110223e-16,1.110223e-16,True
8,diagnosis,2,M_A-resQ,101,0.1000,10.0,1.110223e-16,1.110223e-16,True
12,diagnosis,3,M_A,97,0.1000,NaN,1.110223e-16,1.110223e-16,True
13,diagnosis,3,M_A+Q,97,0.1000,NaN,1.110223e-16,1.110223e-16,True
14,diagnosis,3,M_A-resQ,97,0.1000,10.0,1.110223e-16,1.110223e-16,True
18,diagnosis,4,M_A,92,0.1000,NaN,1.110223e-16,1.110223e-16,True


,task,outer_repeat,outer_fold,training_participants,heldout_participants,reference_recording_count,reference_subject_count,reference_sha256,reference_file,reference_file_sha256,heldout_subject_overlap_n
0,diagnosis,1,1,161,38,385,161,d7ae5797ff8b33200ef6e7334a081c1a39c84236267549...,outputs\goal3\stageD_goal2_model_bundle_bridge...,38f3ac4e1b9dcde2c253b455dafe537ca81fa8ca5b652a...,0
2,diagnosis,1,2,158,41,382,158,be2af023812a8be274798530dcbbd30eaf66fa3486cb42...,outputs\goal3\stageD_goal2_model_bundle_bridge...,ab5006349f3a6594a9ac85b87ca662583a04ddad133ad0...,0
4,diagnosis,1,3,159,40,386,159,fbe7c1242e45cafff9c28c0a40e324f7c0b6a822aba49c...,outputs\goal3\stageD_goal2_model_bundle_bridge...,79d24991f65c82fc2e7079780326523c1de762ddda8727...,0
6,diagnosis,1,4,158,41,391,158,83d61cefdcc0b3433992b370128e2843c7c354590c9427...,outputs\goal3\stageD_goal2_model_bundle_bridge...,e2e84d640f0339b2c14687ce8c6dd79ab86aa2db1254f6...,0
8,diagnosis,1,5,160,39,388,160,237862daec84df8f588a68efb3d6add089cc41a1187507...,outputs\goal3\stageD_goal2_model_bundle_bridge...,ebc3c304e89763f8f58408c5ac29beff7d5a28ba3a59f9...,0
1,severity,1,1,116,29,321,116,a7a94020d7a780f6bba7f70048c88758440f90eca48f62...,outputs\goal3\stageD_goal2_model_bundle_bridge...,0a6159e618288ccaab8595341d95dc2eb3c57478648579...,0
3,severity,1,2,118,27,323,118,0c8911c74244a1214d656bc97db7cb8f331d25babdd527...,outputs\goal3\stageD_goal2_model_bundle_bridge...,988925ace7c3bd47bd9da37d83de7861b550712b5f78e1...,0
5,severity,1,3,117,28,323,117,ddfe38001c8f91fc9d00da4f020317c409c0aaa6826319...,outputs\goal3\stageD_goal2_model_bundle_bridge...,5976d8ea0ef768db1ba762b051b77ad586476282e4bee1...,0
7,severity,1,4,114,31,328,114,649280eb890a07029a760df99942258798658cc2895dab...,outputs\goal3\stageD_goal2_model_bundle_bridge...,7fa24399f793b9b80cb6e7f654326a5952cedcb03e963a...,0
9,severity,1,5,115,30,321,115,ae872c69a5e6ec98abacb4945de351461162ce4e79b7bb...,outputs\goal3\stageD_goal2_model_bundle_bridge...,8c66cc4aa00fb673f620c682c0ac9937ca842b9b2995ba...,0


GOAL 3 GOAL-2 MODEL BUNDLE BRIDGE: PASS
Seal: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageD_goal2_model_bundle_bridge_v1_0\final\GOAL3_GOAL2_MODEL_BUNDLE_SEAL.json
Five bundle files:
   C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageD_goal2_model_bundle_bridge_v1_0\final\bundles\goal2_outer_repeat1_fold_1.joblib
   C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageD_goal2_model_bundle_bridge_v1_0\final\bundles\goal2_outer_repeat1_fold_2.joblib
   C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageD_goal2_model_bundle_bridge_v1_0\final\bundles\goal2_outer_repeat1_fold_3.joblib
   C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageD_goal2_model_bundle_bridge_v1_0\final\bundles\goal2_outer_repeat1_fold_4.joblib
   C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageD_goal2_model_bundle_bridge_v1_0\fin


## What to send back after this notebook runs

Please send the final output from the seal cell, especially:

- the 30-row reproduction summary;
- the 10-row QCHAN reference audit;
- the reported **maximum absolute post-serialization prediction difference**;
- the final `GOAL 3 GOAL-2 MODEL BUNDLE BRIDGE: PASS` line.

Do **not** start the full perturbation experiment if this notebook raises any error. A failed reproduction is a model-provenance problem to resolve, not a tolerance to relax.
